In [20]:
import pandas as pd
import unicodedata
import re
import numpy as np
from transformers import EarlyStoppingCallback
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [21]:
train_df=pd.read_csv("/kaggle/input/datasets/poushalisanyal/cardiff-nlp/merged_train.csv")
test_df=pd.read_csv("/kaggle/input/datasets/poushalisanyal/cardiff-nlp/merged_test.csv")
valid_df=pd.read_csv("/kaggle/input/datasets/poushalisanyal/cardiff-nlp/merged_valid.csv")

In [22]:
train_df.head()

,text,label,source,lang
0,RT @user: @user @user وصلنا لاقتصاد اسوء من ...,negative,sem_eval_2017,ar
1,كاني ويست، دريك، نيكي، بيونسيه، قاقا http,neutral,sem_eval_2017,ar
2,@user على فكره شركة محترمه حداعطوني كيبل كهديه...,positive,sem_eval_2017,ar
3,RT @user: المتعه افضل من الزواج لهتك اعراض عام...,negative,sem_eval_2017,ar
4,القوات البرية السعودية والقوات الفرنسية الخاصة...,neutral,sem_eval_2017,ar


In [23]:
train_df.shape

(30399, 4)

In [24]:
labels = sorted(train_df["label"].unique())

label2id = {v: i for i, v in enumerate(labels)}
id2label = {i: v for v, i in label2id.items()}

train_df["label"] = train_df["label"].map(label2id)
valid_df["label"] = valid_df["label"].map(label2id)
test_df["label"] = test_df["label"].map(label2id)

In [25]:
def normalize(text):
    return unicodedata.normalize("NFKC", str(text))

def remove_noise(text):
    text = re.sub(r"http\S+|www\S+", "", text)   # links
    text = re.sub(r"@\w+", "", text)             # mentions
    text = re.sub(r"\s+", " ", text).strip()     # spaces
    return text

def fix_punct(text):
    return re.sub(r"([!?.,])\1+", r"\1", text)

def fix_repeating(text):
    return re.sub(r"(.)\1{2,}", r"\1\1", text)

def remove_emojis(text):
    emoji_pattern = re.compile("["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
    "]+", flags=re.UNICODE)
    return emoji_pattern.sub("", text)

In [26]:
train_df["text"] = train_df["text"].apply(remove_noise)
train_df["text"] = train_df["text"].apply(normalize)
train_df["text"] = train_df["text"].apply(fix_punct)
train_df["text"] = train_df["text"].apply(fix_repeating)
train_df["text"] = train_df["text"].apply(remove_emojis)

In [27]:
test_df["text"] = test_df["text"].apply(remove_noise)
test_df["text"] = test_df["text"].apply(normalize)
test_df["text"] = test_df["text"].apply(fix_punct)
test_df["text"] = test_df["text"].apply(fix_repeating)
test_df["text"] = test_df["text"].apply(remove_emojis)

In [28]:
valid_df["text"] = valid_df["text"].apply(remove_noise)
valid_df["text"] = valid_df["text"].apply(normalize)
valid_df["text"] = valid_df["text"].apply(fix_punct)
valid_df["text"] = valid_df["text"].apply(fix_repeating)
valid_df["text"] = valid_df["text"].apply(remove_emojis)

In [29]:
model_name = "cardiffnlp/twitter-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [30]:
def tokenize_fn(ex):
    return tokenizer(
        ex["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [31]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))
val_ds = Dataset.from_pandas(valid_df.reset_index(drop=True))

train_tokd = train_ds.map(tokenize_fn, batched=True)
test_tokd = test_ds.map(tokenize_fn, batched=True)
val_tokd = val_ds.map(tokenize_fn, batched=True)

Map:   0%|          | 0/30399 [00:00<?, ? examples/s]

Map:   0%|          | 0/8465 [00:00<?, ? examples/s]

Map:   0%|          | 0/4857 [00:00<?, ? examples/s]

In [32]:
num_labels = 3
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# increase dropout
model.config.hidden_dropout_prob = 0.3
model.config.attention_probs_dropout_prob = 0.3

# freeze embedding layer
for param in model.roberta.embeddings.parameters():
    param.requires_grad = False

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.decoder.weight      | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [33]:
!pip install evaluate
import evaluate

In [34]:
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels, average="weighted")["precision"],
        "recall": recall.compute(predictions=preds, references=labels, average="weighted")["recall"],
        "f1": f1.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }

In [35]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [36]:
training_args = TrainingArguments(
    output_dir="sentiment_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    learning_rate=2e-5,
    lr_scheduler_type="linear",
    warmup_steps=0.1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=6,
    weight_decay=0.01,
    label_smoothing_factor=0.0,
    max_grad_norm=1.0,
    fp16=True,
    save_total_limit=2
)

In [37]:
trainer = Trainer(model=model,
    args=training_args,
    train_dataset=train_tokd,
    eval_dataset=val_tokd,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [38]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,1.677716,0.599959,0.593121,0.599959,0.591818
2,3.605680,1.565378,0.626724,0.630661,0.626724,0.611701
3,2.900927,1.522560,0.639489,0.644364,0.639489,0.640196
4,2.526428,1.584325,0.647931,0.645034,0.647931,0.646124
5,2.235221,1.591285,0.651019,0.647210,0.651019,0.647971
6,1.986599,1.648940,0.653490,0.652042,0.653490,0.652380


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=2850, training_loss=2.5576188365200108, metrics={'train_runtime': 2659.4115, 'train_samples_per_second': 68.584, 'train_steps_per_second': 1.072, 'total_flos': 1.1997577178270208e+16, 'train_loss': 2.5576188365200108, 'epoch': 6.0})

In [39]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 1.648939609527588,
 'eval_accuracy': 0.6534898085237801,
 'eval_precision': 0.6520422117763083,
 'eval_recall': 0.6534898085237801,
 'eval_f1': 0.6523803022750659,
 'eval_runtime': 24.2805,
 'eval_samples_per_second': 200.037,
 'eval_steps_per_second': 6.26,
 'epoch': 6.0}

In [43]:
test_results = trainer.evaluate(test_tokd)
print(test_results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 1.9248589277267456, 'eval_accuracy': 0.6086237448316598, 'eval_precision': 0.6095827770011318, 'eval_recall': 0.6086237448316598, 'eval_f1': 0.6084465982565509, 'eval_runtime': 41.9947, 'eval_samples_per_second': 201.573, 'eval_steps_per_second': 6.31, 'epoch': 6.0}


In [44]:
valid_results = trainer.evaluate(val_tokd)
print(valid_results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 1.648939609527588, 'eval_accuracy': 0.6534898085237801, 'eval_precision': 0.6520422117763083, 'eval_recall': 0.6534898085237801, 'eval_f1': 0.6523803022750659, 'eval_runtime': 24.4186, 'eval_samples_per_second': 198.905, 'eval_steps_per_second': 6.225, 'epoch': 6.0}
